In [1]:
import great_expectations as gx
import pandas as pd
import os

df = pd.read_csv("../data/pakwheels.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (62302, 14)
Columns: ['addref', 'city', 'assembly', 'body', 'make', 'model', 'year', 'engine', 'transmission', 'fuel', 'color', 'registered', 'mileage', 'price']


,addref,city,assembly,body,make,model,year,engine,transmission,fuel,color,registered,mileage,price
0,7642697,Islamabad,NaN,Compact SUV,KIA,Sorento,2021.0,3500.0,Automatic,Petrol,NaN,Islamabad,54654,9300000.0
1,7909608,Sadiqabad,Imported,Hatchback,Daihatsu,Mira,2020.0,660.0,Automatic,Petrol,White,Punjab,10000,3700000.0
2,7929224,Peshawar,Imported,Hatchback,Toyota,Vitz,2018.0,1000.0,Automatic,Petrol,Silver,Islamabad,123000,4150000.0
3,7832366,Lahore,NaN,Sedan,Toyota,Corolla,2019.0,1600.0,Automatic,Petrol,White,Lahore,105000,4850000.0
4,7866725,Islamabad,NaN,Sedan,Toyota,Corolla,2022.0,1600.0,Automatic,Petrol,White,Islamabad,6500,6600000.0


In [2]:
print("NULL counts per column:")
print(df.isnull().sum())

NULL counts per column:
addref              0
city                0
assembly        42928
body             7091
make                0
model               0
year             3786
engine              3
transmission        0
fuel              709
color            1192
registered          0
mileage             0
price             472
dtype: int64


In [3]:
df[['year','engine','mileage','price']].describe()


,year,engine,mileage,price
count,58516.000000,62299.000000,62302.000000,6.183000e+04
mean,2012.843991,1406.731887,91043.031476,3.886141e+06
std,7.497370,706.209678,89279.502257,5.452661e+06
min,1990.000000,3.000000,1.000000,1.100000e+05
25%,2007.000000,1000.000000,34000.000000,1.450000e+06
50%,2015.000000,1300.000000,80000.000000,2.700000e+06
75%,2019.000000,1600.000000,123456.000000,4.500000e+06
max,2022.000000,15000.000000,1000000.000000,1.650000e+08


In [4]:
print("transmission:", df['transmission'].unique())
print("fuel:", df['fuel'].unique())
print("assembly:", df['assembly'].unique())
print("\nbody types:")
print(df['body'].value_counts())

transmission: ['Automatic' 'Manual']
fuel: ['Petrol' 'Hybrid' 'Diesel' nan]
assembly: [nan 'Imported']

body types:
body
Sedan                24176
Hatchback            20036
SUV                   4072
Crossover             1730
Mini Van              1067
Compact sedan          630
Double Cabin           623
MPV                    616
Van                    577
Micro Van              430
Pick Up                411
Compact SUV            389
Station Wagon          181
Coupe                   69
Truck                   68
High Roof               62
Convertible             38
Single Cabin            18
Off-Road Vehicles        9
Mini Vehicles            5
Compact hatchback        4
Name: count, dtype: int64


In [ ]:
from great_expectations.core.expectation_suite import ExpectationSuite
from great_expectations.expectations import (
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeUnique,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToBeInSet,
    ExpectColumnValuesToBeOfType,
    ExpectTableRowCountToBeBetween,
)

context = gx.get_context()
suite = ExpectationSuite(name="pakwheels_suite")

# 1. COMPLETENESS 
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="addref"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="city"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="make"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="model"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="transmission"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="registered"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="mileage"))

# 2. UNIQUENESS 
suite.add_expectation(ExpectColumnValuesToBeUnique(column="addref"))

# 3. YEAR RANGE 
suite.add_expectation(ExpectColumnValuesToBeBetween(
    column="year", min_value=1990, max_value=2023
))

# 4. ENGINE SIZE
suite.add_expectation(ExpectColumnValuesToBeBetween(
    column="engine", min_value=660, max_value=5000
))

# 5. MILEAGE RANGE  100km minimum, 500,000km maximum
suite.add_expectation(ExpectColumnValuesToBeBetween(
    column="mileage", min_value=100, max_value=500000
))

# 6. PRICE RANGE in PKR  200,000 to 50,000,000
suite.add_expectation(ExpectColumnValuesToBeBetween(
    column="price", min_value=200000, max_value=50000000
))

# 7. VALID TRANSMISSION VALUES
suite.add_expectation(ExpectColumnValuesToBeInSet(
    column="transmission", value_set=["Automatic", "Manual"]
))

# 8. VALID FUEL TYPES
suite.add_expectation(ExpectColumnValuesToBeInSet(
    column="fuel", value_set=["Petrol", "Diesel", "Hybrid", "CNG", "Electric", "LPG"]
))

# 9. VALID BODY TYPES 
suite.add_expectation(ExpectColumnValuesToBeInSet(
    column="body",
    value_set=[
        "Compact SUV", "Hatchback", "Sedan", "Crossover", "SUV",
        "Compact sedan", "Micro Van", "MPV", "Van", "Pick Up",
        "Mini Van", "Double Cabin", "Station Wagon", "Convertible",
        "Coupe", "Off-Road Vehicles", "High Roof", "Truck",
        "Single Cabin", "Mini Vehicles", "Compact hatchback"
    ]
))

# 10. VALID ASSEMBLY VALUES
suite.add_expectation(ExpectColumnValuesToBeInSet(
    column="assembly", value_set=["Imported", "Local"]
))

# 11. DATA TYPE CHECKS
suite.add_expectation(ExpectColumnValuesToBeOfType(column="mileage", type_="int64"))
suite.add_expectation(ExpectColumnValuesToBeOfType(column="engine",  type_="float64"))

# 12. ROW COUNT
suite.add_expectation(ExpectTableRowCountToBeBetween(min_value=60000, max_value=100000))

print(f"Suite built with {len(suite.expectations)} expectations")

Suite built with 19 expectations


In [6]:
ds       = context.data_sources.add_or_update_pandas(name="pakwheels")
asset    = ds.add_dataframe_asset(name="pakwheels_asset")
batch_def = asset.add_batch_definition_whole_dataframe("pakwheels_batch")
batch    = batch_def.get_batch(batch_parameters={"dataframe": df})

results  = batch.validate(suite)

total  = len(results.results)
passed = sum(1 for r in results.results if r.success)
failed = total - passed

print(f"Total  : {total}")
print(f"Passed : {passed} ")
print(f"Failed : {failed} ")
print(f"Overall: {'PASSED' if results.success else 'FAILED'}")

Calculating Metrics:   0%|          | 0/100 [00:00<?, ?it/s]

Total  : 19
Passed : 16 
Failed : 3 
Overall: FAILED


In [8]:
print("FAILED CHECKS WITH DETAILS:")
print("="*60)
for r in results.results:
    if not r.success:
        exp_type = r.expectation_config.type
        col      = r.expectation_config.kwargs.get("column", "TABLE-LEVEL")
        uc = r.result.get("unexpected_count", "N/A")
        up = r.result.get("unexpected_percent", 0)
        mc = r.result.get("missing_count", "N/A")
        print(f"\n Column  : {col}")
        print(f"  Check   : {exp_type}")
        if uc != "N/A":
            print(f"  Bad rows: {uc}  ({round(up,2)}%)")
        if mc not in ("N/A", 0):
            print(f"  Nulls   : {mc}")

FAILED CHECKS WITH DETAILS:

 Column  : mileage
  Check   : expect_column_values_to_be_between
  Bad rows: 2453  (3.94%)

 Column  : engine
  Check   : expect_column_values_to_be_between
  Bad rows: 455  (0.73%)
  Nulls   : 3

 Column  : price
  Check   : expect_column_values_to_be_between
  Bad rows: 209  (0.34%)
  Nulls   : 472


In [10]:
os.makedirs("../reports", exist_ok=True)

with open("../reports/pakwheels_validation_report.txt", "w", encoding="utf-8") as f:
    f.write("="*55 + "\n")
    f.write("  DATA QUALITY REPORT — PAKWHEELS DATASET\n")
    f.write("="*55 + "\n\n")
    f.write(f"Overall Result : {'PASSED' if results.success else 'FAILED'}\n")
    f.write(f"Total Checks   : {total}\n")
    f.write(f"Passed         : {passed}\n")
    f.write(f"Failed         : {failed}\n\n")
    f.write("─"*55 + "\n")
    f.write("  DETAILED RESULTS\n")
    f.write("─"*55 + "\n\n")

    for r in results.results:
        status   = "PASS" if r.success else "FAIL"
        exp_type = r.expectation_config.type
        col      = r.expectation_config.kwargs.get("column", "TABLE-LEVEL")
        f.write(f"[{status}]  Column: {col}\n")
        f.write(f"         Check : {exp_type}\n")
        if not r.success and r.result:
            uc = r.result.get("unexpected_count", "N/A")
            up = r.result.get("unexpected_percent", 0)
            mc = r.result.get("missing_count", "N/A")
            if uc != "N/A":
                f.write(f"         Bad rows  : {uc}\n")
                f.write(f"         Bad %     : {round(up,2)}%\n")
            if mc not in ("N/A", 0):
                f.write(f"         Null rows : {mc}\n")
        f.write("\n")

print("Report saved → reports/pakwheels_validation_report.txt")

Report saved → reports/pakwheels_validation_report.txt
